In [1]:
import pandas as pd
import numpy as np
import re 

In [2]:
import urllib.request
url = """https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"""
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x1dc1abc2030>)

In [3]:
with open("the-verdict.txt", "r", encoding='utf-8') as f:
    raw_text = f.read()

In [4]:
raw_text[:500]

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\''

# Tokenization Based on Space

In [5]:
class SpaceTokenizer:
    class Dictionary:
        def __init__(self):
            self.vocab = {}
        
        def makeDictionary(self, text):
            preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
            preprocessed = [item.strip() for item in preprocessed if item.strip() ]
            all_words = sorted(set(preprocessed))
            self.vocab = {token:integer for integer, token in enumerate(all_words) }
            self.vocab['<|unk|>'] = len(self.vocab)
            print("Vocab Size: ",len(self.vocab))
            return self.vocab

    def __init__(self, train_text):
        dic = SpaceTokenizer.Dictionary()
        self.str_to_int = dic.makeDictionary(train_text)
        self.int_to_str = {y:x for x,y in self.str_to_int.items()}
        
    # word -> token_id
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [ item.strip() for item in preprocessed if item.strip() ]
        preprocessed = [item if item in self.str_to_int 
                        else "<|unk|>" for item in preprocessed]
        ids =  [self.str_to_int[word] for word in preprocessed]
        return ids
    
    # token_id -> word
    def decode(self, ids):
        text = " ".join(self.int_to_str[id] for id in ids)
        text = re.sub(r'\s+([,.:;?!"()\'])',r'\1',text)
        return text

In [6]:
tokenize = SpaceTokenizer(raw_text)

Vocab Size:  1131


In [19]:
print("Vocab Size:", len(tokenize.str_to_int))

Vocab Size: 1131


In [7]:
ids = tokenize.encode("I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow")
ids

[53, 44, 149, 1003, 57, 38, 818, 115, 256, 486, 6, 1002, 115, 500, 435]

In [8]:
text_back = tokenize.decode(ids)
text_back

'I HAD always thought Jack Gisburn rather a cheap genius -- though a good fellow'

# Dataset Build

In [13]:
import torch
from torch.utils.data import Dataset, DataLoader


class DatasetSpaceToken(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(text)
        
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            
            target_chunk = token_ids[i+1 : i+1+max_length]
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]
            

In [14]:
def create_dataloader(text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = SpaceTokenizer(text)
    dataset = DatasetSpaceToken(text, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [15]:
dataloader = create_dataloader(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False
)

Vocab Size:  1131


In [16]:
data_iter = iter(dataloader)
for i, batch in enumerate(data_iter):
    print(f"Batch {i}:", batch)
    print("-------------")

Batch 0: [tensor([[  53,   44,  149, 1003],
        [  57,   38,  818,  115],
        [ 256,  486,    6, 1002],
        [ 115,  500,  435,  392],
        [   6,  908,  585, 1077],
        [ 709,  508,  961, 1016],
        [ 663, 1016,  535,  987],
        [   5,  568,  988,  538]]), tensor([[  44,  149, 1003,   57],
        [  38,  818,  115,  256],
        [ 486,    6, 1002,  115],
        [ 500,  435,  392,    6],
        [ 908,  585, 1077,  709],
        [ 508,  961, 1016,  663],
        [1016,  535,  987,    5],
        [ 568,  988,  538,  722]])]
-------------
Batch 1: [tensor([[ 722,  549,  496,    5],
        [ 533,  514,  370,  549],
        [ 748,    5,  661,  115],
        [ 841, 1102,    5,  157],
        [ 397,  547,  568,  115],
        [1066,  727,  988,   84],
        [   7,    3,   99,   53],
        [ 818, 1003,  585, 1120]]), tensor([[ 549,  496,    5,  533],
        [ 514,  370,  549,  748],
        [   5,  661,  115,  841],
        [1102,    5,  157,  397],
        

# Token Embedding

In [20]:
vocab_size = len(tokenize.str_to_int)
output_dim = 256
print(vocab_size, output_dim)

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

1131 256


In [21]:
input_data_first_batch = None
target_data_first_batch = None

for input_data, output_fetch in dataloader:
    input_data_first_batch, target_data_first_batch = input_data, output_fetch
    break

In [22]:
token_embedding_first_batch = token_embedding_layer(input_data_first_batch)
token_embedding_first_batch

tensor([[[ 1.0168e+00,  7.5423e-01, -8.7132e-02,  ...,  8.4368e-01,
          -5.5376e-01,  1.8337e+00],
         [ 6.1233e-04, -8.4291e-01, -2.6628e-01,  ..., -1.0844e-01,
           8.7488e-02,  1.3467e+00],
         [-5.0620e-01, -9.2669e-01,  9.6990e-01,  ...,  2.2269e+00,
          -2.6779e-01,  1.9588e+00],
         [ 1.4274e+00, -1.4432e-01,  1.0979e+00,  ..., -3.8653e-01,
           8.9011e-01, -8.0100e-01]],

        [[ 8.0543e-02, -5.1508e-01,  7.7537e-02,  ..., -1.1005e+00,
          -1.0964e+00, -7.6473e-01],
         [ 5.3073e-02,  7.7904e-02,  2.1850e+00,  ..., -3.2513e-01,
          -1.0578e-02, -1.1111e+00],
         [-4.7588e-01, -4.8873e-01, -1.3790e+00,  ..., -7.8818e-01,
           9.1792e-03,  1.3654e+00],
         [-7.3667e-02, -1.9649e-01,  3.4160e-01,  ...,  5.0375e-02,
          -2.1589e+00,  4.3347e-01]],

        [[-1.0039e-01, -4.1604e-02,  6.5816e-01,  ...,  2.2585e-01,
           3.2438e-01,  1.7552e+00],
         [ 8.6796e-01,  7.0057e-01,  7.1259e-01,  .

In [24]:
print(input_data_first_batch.shape)
print(token_embedding_first_batch.shape)

torch.Size([8, 4])
torch.Size([8, 4, 256])
